In [0]:
import subprocess
import os

def run_cmd(cmd, cwd=None):
    result = subprocess.run(
        cmd, shell=True, capture_output=True, text=True, cwd=cwd
    )
    print(f"CMD: {cmd}")
    print(f"RC: {result.returncode}")
    if result.stdout: print(f"OUT: {result.stdout}")
    if result.stderr: print(f"ERR: {result.stderr}")
    return result.returncode

run_cmd("rm -rf /tmp/energy_dbt")
run_cmd("git clone https://github.com/EnochBabs/de-portfolio /tmp/energy_dbt")
run_cmd("pip install dbt-databricks --quiet")

host = "adb-7405605894821381.1.azuredatabricks.net"
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
http_path = "/sql/protocolv1/o/7405605894821381/0405-211727-1i4q0tmo"

profiles_dir = "/tmp/dbt_profiles"
os.makedirs(profiles_dir, exist_ok=True)

profiles = f"""energy_pipeline:
  target: dev
  outputs:
    dev:
      type: databricks
      host: {host}
      http_path: {http_path}
      token: {token}
      catalog: energy
      schema: gold
      threads: 4
"""

with open(f"{profiles_dir}/profiles.yml", "w") as f:
    f.write(profiles)
print("profiles.yml written")

rc = run_cmd(
    f"dbt run --project-dir /tmp/energy_dbt/energy/dbt --profiles-dir {profiles_dir}",
    cwd="/tmp/energy_dbt/energy/dbt"
)
if rc != 0:
    raise Exception("dbt run failed")

rc = run_cmd(
    f"dbt test --project-dir /tmp/energy_dbt/energy/dbt --profiles-dir {profiles_dir}",
    cwd="/tmp/energy_dbt/energy/dbt"
)
if rc != 0:
    raise Exception("dbt test failed")

print("Pipeline complete")